In [1]:
# Import core libraries and the shared scorecard functions from src/
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from scorecard_utils import (
    calculate_woe_iv, calculate_woe_iv_categorical, collapse_rare_categories,
    merge_missing_into_nearest_bin, calculate_woe_iv_with_zero_bin,
    extract_edges, ordered_bin_woe, apply_frozen_woe
)

train_bureau = pd.read_csv('../data/processed/train_bureau.csv')
print(train_bureau.shape)

(307511, 156)


In [2]:
# Rebuild the exact same train/validation split used in 04_scorecard_woe.ipynb
# (same 17 final variables, same random_state), to reproduce the frozen
# pipeline from scratch as a validation of reproducibility.

FINAL_17_VARS = [
    'EXT_SOURCE_2', 'EXT_SOURCE_3', 'NAME_EDUCATION_TYPE_BINNED', 'EXT_SOURCE_1',
    'AMT_CREDIT', 'CODE_GENDER', 'YEARS_EMPLOYED', 'ORGANIZATION_TYPE_BINNED',
    'FLOORSMAX_AVG', 'BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT', 'BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN',
    'OCCUPATION_TYPE_BINNED', 'REGION_POPULATION_RELATIVE', 'DAYS_ID_PUBLISH',
    'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN', 'DAYS_LAST_PHONE_CHANGE', 'BUREAU_DAYS_CREDIT_MEAN',
]

RAW_SOURCE = {
    'NAME_EDUCATION_TYPE_BINNED': 'NAME_EDUCATION_TYPE',
    'ORGANIZATION_TYPE_BINNED': 'ORGANIZATION_TYPE',
    'OCCUPATION_TYPE_BINNED': 'OCCUPATION_TYPE',
}

X = pd.DataFrame(index=train_bureau.index)
for var in FINAL_17_VARS:
    X[var] = train_bureau[RAW_SOURCE.get(var, var)]
X['SK_ID_CURR'] = train_bureau['SK_ID_CURR']
y = train_bureau['TARGET']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape, "| Val:", X_val.shape)
print("TARGET rate (train):", y_train.mean().round(4), "| (val):", y_val.mean().round(4))

Train: (246008, 18) | Val: (61503, 18)
TARGET rate (train): 0.0807 | (val): 0.0807


In [3]:
# Check zero-concentration for BUREAU_AMT_CREDIT_SUM_DEBT_MEAN specifically,
# since it replaced the originally zero-corrected _SUM variant during
# correlation pruning and was never individually validated for this pattern.
n_missing = train_bureau['BUREAU_AMT_CREDIT_SUM_DEBT_MEAN'].isnull().sum()
n_zero = (train_bureau['BUREAU_AMT_CREDIT_SUM_DEBT_MEAN'] == 0).sum()
n_valid = train_bureau['BUREAU_AMT_CREDIT_SUM_DEBT_MEAN'].notnull().sum()
print(f"Missing: {n_missing} ({n_missing/len(train_bureau)*100:.1f}%)")
print(f"Zero (among non-null): {n_zero} ({n_zero/n_valid*100:.1f}%)")

Missing: 51380 (16.7%)
Zero (among non-null): 69689 (27.2%)


In [5]:
# Compare standard WoE vs. zero-isolated WoE for BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,
# now confirmed to have 27.2% zero concentration among non-null values,
# a pattern similar to BUREAU_AMT_CREDIT_SUM_DEBT_SUM (Group 4, 29.2%),
# which received this treatment originally; this variable did not.

train_xy = X_train.copy()
train_xy['TARGET'] = y_train

standard_result, standard_iv = calculate_woe_iv(train_xy, 'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN', min_bad_per_bin=0)
zero_result, zero_iv = calculate_woe_iv_with_zero_bin(train_xy, 'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN', min_bad_per_bin=0)

print("=== Padrão (qcut simples) ===")
print(standard_result.round(4).to_string(index=False))
print(f"IV total: {standard_iv:.4f}\n")

print("=== Zero isolado ===")
print(zero_result.round(4).to_string(index=False))
print(f"IV total: {zero_iv:.4f}")

=== Padrão (qcut simples) ===
                       bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
(-1083614.6709999999, 0.0]    56647   3126    0.2367   0.1574  0.4079        0.0323
           (0.0, 4181.063]     4810    304    0.0199   0.0153  0.2624        0.0012
       (4181.063, 22444.5]    20486   1438    0.0842   0.0724  0.1511        0.0018
       (22444.5, 44196.06]    20485   1611    0.0835   0.0811  0.0284        0.0001
      (44196.06, 72305.55]    20486   1828    0.0825   0.0920 -0.1094        0.0010
     (72305.55, 112972.95]    20485   1883    0.0823   0.0948 -0.1421        0.0018
     (112972.95, 182056.5]    20486   2040    0.0816   0.1027 -0.2306        0.0049
    (182056.5, 348749.994]    20485   1999    0.0817   0.1007 -0.2081        0.0039
  (348749.994, 43650000.0]    20486   1673    0.0832   0.0842 -0.0126        0.0000
                   Missing    41152   3958    0.1645   0.1993 -0.1920        0.0067
IV total: 0.0537

=== Zero isolado ===
       

## Note: BUREAU_AMT_CREDIT_SUM_DEBT_MEAN zero-concentration check

This variable entered the final 17 via correlation pruning (replacing
BUREAU_AMT_CREDIT_SUM_DEBT_SUM, which did receive the Group 4 zero-isolation
treatment), and was never individually checked for the same pattern.
Confirmed 27.2% zero concentration among non-null values, similar to the
_SUM variant (29.2%). Tested zero-isolated WoE against the standard qcut
treatment: IV 0.0542 vs. 0.0537, a negligible difference. Kept the standard
treatment; the plain quantile binning already happened to isolate zero
values reasonably well in this specific variable's distribution.

In [6]:
# Freeze bin/category structures and WoE values for all 17 final variables,
# using X_train/y_train only. EXT_SOURCE_2 uses the Missing-merge treatment
# (Group 1); all others use plain calculate_woe_iv / calculate_woe_iv_categorical,
# confirmed sufficient for this variable set (BUREAU_AMT_CREDIT_SUM_DEBT_MEAN
# checked and confirmed not to need zero-isolation, see note above).

MISSING_MERGE_VARS = ['EXT_SOURCE_2']

train_xy = X_train.copy()
train_xy['TARGET'] = y_train

frozen = {}

for var in FINAL_17_VARS:

    if var in MISSING_MERGE_VARS:
        result, iv = merge_missing_into_nearest_bin(train_xy, var)
        edges = extract_edges(result)
        bin_woe_ordered = ordered_bin_woe(result)

        is_missing = train_xy[var].isnull()
        total_good = (train_xy['TARGET'] == 0).sum()
        total_bad = (train_xy['TARGET'] == 1).sum()
        n_bad_missing = train_xy.loc[is_missing, 'TARGET'].sum()
        n_good_missing = is_missing.sum() - n_bad_missing
        missing_woe = np.log(
            ((n_good_missing + 0.5) / (total_good + 0.5)) /
            ((n_bad_missing + 0.5) / (total_bad + 0.5))
        )
        nearest_woe = min(bin_woe_ordered, key=lambda w: abs(w - missing_woe))
        frozen[var] = {'type': 'numeric', 'edges': edges, 'bin_woe_ordered': bin_woe_ordered,
                        'zero_woe': None, 'missing_woe': nearest_woe}

    elif var in ['NAME_EDUCATION_TYPE_BINNED', 'ORGANIZATION_TYPE_BINNED', 'OCCUPATION_TYPE_BINNED']:
        raw_col = RAW_SOURCE[var]
        raw_series = X_train[var]
        collapsed_series = collapse_rare_categories(train_xy.assign(**{raw_col: raw_series}), raw_col)
        category_map = dict(zip(raw_series.astype('object').fillna('Missing'), collapsed_series))
        temp_df = pd.DataFrame({var: collapsed_series, 'TARGET': y_train})
        result, iv = calculate_woe_iv_categorical(temp_df, var, min_bad_per_bin=0)
        frozen[var] = {'type': 'categorical', 'table': result, 'category_map': category_map}

    elif pd.api.types.is_numeric_dtype(X_train[var]):
        result, iv = calculate_woe_iv(train_xy, var, min_bad_per_bin=0)
        edges = extract_edges(result)
        bin_woe_ordered = ordered_bin_woe(result)
        missing_rows = result[result['bin'] == 'Missing']
        frozen[var] = {'type': 'numeric', 'edges': edges, 'bin_woe_ordered': bin_woe_ordered,
                        'zero_woe': None,
                        'missing_woe': missing_rows['woe'].values[0] if len(missing_rows) else None}

    else:
        result, iv = calculate_woe_iv_categorical(train_xy, var, min_bad_per_bin=0)
        frozen[var] = {'type': 'categorical', 'table': result, 'category_map': None}

print(f"Frozen structures built for {len(frozen)} of {len(FINAL_17_VARS)} variables.")

AVISO [EXT_SOURCE_2]: 1 bin(s) com menos de 100 casos 'bad', WoE pode ser instável:
    bin  n_bad
Missing     41

'EXT_SOURCE_2': Missing (WoE=0.0375) fundido com bin (0.512, 0.566] (WoE original=0.0847)
'NAME_EDUCATION_TYPE': Other_grouped ainda instável, fundido com 'Higher education'
Frozen structures built for 17 of 17 variables.


In [7]:
# Transform X_train and X_val into WoE space using only the frozen
# structures above, pure lookup, nothing recalculated from X_val.

X_train_woe = pd.DataFrame(index=X_train.index)
X_val_woe = pd.DataFrame(index=X_val.index)

for var in FINAL_17_VARS:
    X_train_woe[var] = apply_frozen_woe(X_train[var], frozen[var])
    X_val_woe[var] = apply_frozen_woe(X_val[var], frozen[var])

print("X_train_woe:", X_train_woe.shape, "| X_val_woe:", X_val_woe.shape)
print("Nulls (train):", X_train_woe.isnull().sum().sum(), "| Nulls (val):", X_val_woe.isnull().sum().sum())

X_train_woe: (246008, 17) | X_val_woe: (61503, 17)
Nulls (train): 0 | Nulls (val): 0


In [8]:
X_train_woe.head(2)

,EXT_SOURCE_2,EXT_SOURCE_3,NAME_EDUCATION_TYPE_BINNED,EXT_SOURCE_1,AMT_CREDIT,CODE_GENDER,YEARS_EMPLOYED,ORGANIZATION_TYPE_BINNED,FLOORSMAX_AVG,BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT,BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN,OCCUPATION_TYPE_BINNED,REGION_POPULATION_RELATIVE,DAYS_ID_PUBLISH,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,DAYS_LAST_PHONE_CHANGE,BUREAU_DAYS_CREDIT_MEAN
181648,-0.449285,0.545394,0.433340,-0.049271,0.034827,0.154054,-0.260779,-0.073397,0.416358,0.087677,0.134208,-0.291701,-0.150727,0.281571,-0.109447,-0.156136,0.211081
229245,0.083924,-0.152144,-0.108462,-0.059184,0.196560,-0.250222,-0.351108,0.081116,-0.140620,-0.248603,-0.174112,-0.368124,-0.071707,-0.141896,-0.191957,-0.140761,-0.248488


In [9]:
# Final validation: does the rebuilt pipeline reproduce the numbers already
# published in the README and the technical presentation? This is the real
# test, not just "the code runs without error".

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

# ... (célula de congelamento das 17 variáveis, usando as funções copiadas do 04)

model_check = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
model_check.fit(X_train_woe, y_train)
auc_check = roc_auc_score(y_val, model_check.predict_proba(X_val_woe)[:, 1])

print("AUC reproduzido:", round(auc_check, 4), "| Esperado: 0.7406")
print("Bate?", abs(auc_check - 0.7406) < 0.0005)

AUC reproduzido: 0.7405 | Esperado: 0.7406
Bate? True
